In [46]:
import pandas as pd
import numpy as np

In [59]:
df = pd.read_csv("korean_skincare_raw_scraped.csv")

In [60]:
df.head(10)

,Brand,Product Name,Current Price,Original Price,Discount (%),URL,Category,Skin Concerns,Ingredients,Rating,Review_Count
0,KOREAN SKINCARE,The Tidal Glow Advent Calendar 2026,204.95,204.95,0.0,https://korean-skincare.de/products/korean-ski...,NaN,NaN,NaN,5.00,10.0
1,BEAUTY OF JOSEON,Relief Sun: Rice + Niacinamide SPF 50+ PA++++,12.61,17.90,29.6,https://korean-skincare.de/products/beauty-of-...,Sunscreen,All skin types aespecially those looking for h...,"Water, Dibutyl Adipate, Propanediol, Diethylam...",4.75,7298.0
2,DR.ALTHEA,345 Relief Cream,17.82,22.95,22.4,https://korean-skincare.de/products/dr-althea-...,Moisturizers,"Acne-prone skin, sensitive skin, redness-prone...","Aqua, Melaleuca Alternifolia Leaf Water, Propa...",4.60,2644.0
3,BEAUTY OF JOSEON,Dynasty Cream,16.85,23.95,29.6,https://korean-skincare.de/products/beauty-of-...,General Moisturizer,"Good For Oily Skin,Redness Reducing,Reduces Ir...","Water,Glycerin,Butylene Glycol,Niacinamide,Dip...",4.62,2645.0
4,ANUA,Heartleaf Quercetinol Pore Deep Cleansing Foam,12.38,16.45,24.7,https://korean-skincare.de/products/anua-heart...,Cleansers,"Acne prone skin, oily skin, combination skin","Houttuynia Cordata Flower/Leaf/Stem Water, Gly...",4.55,1356.0
5,BEAUTY OF JOSEON,Revive Eye Serum : Ginseng + Retinal,13.00,17.95,27.6,https://korean-skincare.de/products/beauty-of-...,Eye Moisturizer,"Good For Oily Skin,Redness Reducing,Skin Textu...","Water,Panax Ginseng Root Extract,Glycerin,Dipr...",4.53,2114.0
6,ANUA,Heartleaf Pore Control Cleansing Oil,13.46,19.95,32.5,https://korean-skincare.de/products/anua-heart...,Makeup Remover,"Anti-Aging,Scar Healing,Brightening,Acne Trigg...","Ethylhexyl Palmitate,Sorbeth-30 Tetraoleate,So...",4.50,2302.0
7,SKIN1004,Madagascar Centella Ampoule,8.31,10.95,24.1,https://korean-skincare.de/products/skin1004-m...,Serum,Redness Reducing,"Centella Asiatica Extract,Water,Glycerin,Cellu...",4.66,1687.0
8,BEAUTY OF JOSEON,Glow Serum Propolis+Niacinamide,13.46,13.95,3.5,https://korean-skincare.de/products/beauty-of-...,Serum,"Good For Oily Skin,Redness Reducing,Reduces Ir...","Propolis Extract,Dipropylene Glycol,Glycerin,B...",4.60,2663.0
9,SKIN1004,Madagascar Centella Hyalu-Cica Water-Fit Sun S...,14.75,20.95,29.6,https://korean-skincare.de/products/skin1004-m...,Sunscreen,"Good For Oily Skin,Redness Reducing,Anti-Aging...","Water,Dibutyl Adipate,Propanediol,Diethylamino...",4.77,1182.0


In [61]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2713 entries, 0 to 2712
Data columns (total 11 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Brand           2713 non-null   str    
 1   Product Name    2713 non-null   str    
 2   Current Price   2713 non-null   float64
 3   Original Price  2713 non-null   float64
 4   Discount (%)    2713 non-null   float64
 5   URL             2713 non-null   str    
 6   Category        2709 non-null   str    
 7   Skin Concerns   2314 non-null   str    
 8   Ingredients     2601 non-null   str    
 9   Rating          2713 non-null   float64
 10  Review_Count    2713 non-null   float64
dtypes: float64(5), str(6)
memory usage: 2.4 MB


In [62]:
df.duplicated().value_counts()

False    2713
Name: count, dtype: int64

In [63]:
df.isna().sum()

Brand               0
Product Name        0
Current Price       0
Original Price      0
Discount (%)        0
URL                 0
Category            4
Skin Concerns     399
Ingredients       112
Rating              0
Review_Count        0
dtype: int64

In [64]:
df["Category"].value_counts()

Category
Moisturizers           221
Serums                 200
Face Masks             179
General Moisturizer    169
Toner                  169
                      ... 
Set                      2
Lip Makeup               1
Hair Mists               1
Tools                    1
Hair Sprays              1
Name: count, Length: 63, dtype: int64

In [65]:
# 1. Define non-skincare categories to drop
non_skincare_categories = [
    # Hair
    "Hair Care", "Other Haircare", "Shampoo", "Hair Mists", "Hair Sprays",
    # Makeup & Nails
    "Make-up", "Make-Up", "Face Makeup", "Eye Makeup", "Lip Makeup", "Blushes", "Gel Nails", "Top Coat",
    # Body, Hand & Fragrance
    "Body Care", "Bath & Body", "Hand Care", "Body Mists", "Perfumes",
    # Tools & General
    "Tools & Accessories", "Accessories", "Tools", "K-Beauty"
]

# 2. Filter out non-skincare items
initial_count = len(df)
df = df[~df["Category"].isin(non_skincare_categories)].copy()

print(f"Removed {initial_count - len(df)} non-skincare products.")
print(f"Remaining skincare products: {len(df)}")

Removed 419 non-skincare products.
Remaining skincare products: 2294


In [66]:
category_mapping = {
    # Cleansers & Makeup Removers
    "Face Cleanser": "Cleanser",
    "Cleansers": "Cleanser",
    "Cleanser": "Cleanser",
    "Makeup Remover": "Cleanser",

    # Toners & Essences
    "Toners": "Toner",
    "Toner": "Toner",
    "Essences": "Essence",
    "Essence": "Essence",

    # Serums, Ampoules & Treatments
    "Serum": "Serum & Ampoule",
    "Ampoules": "Serum & Ampoule",
    "Facial Treatment": "Treatment",
    "Treatments": "Treatment",
    "Spot Patches": "Treatment",
    "Multi Balms": "Treatment",
    "Oil": "Treatment",

    # Moisturizers
    "General Moisturizer": "Moisturizer",
    "Moisturisers": "Moisturizer",
    "Day Moisturizer": "Moisturizer",
    "Night Moisturizer": "Moisturizer",
    "Emulsion": "Moisturizer",
    "Emulsions": "Moisturizer",

    # Sun Protection
    "Sunscreen": "Sunscreen",

    # Masks
    "Sheet Mask": "Mask",
    "Wet Mask": "Mask",
    "Overnight Mask": "Mask",
    "Masks": "Mask",

    # Exfoliators
    "Exfoliator": "Exfoliator",
    "Exfoliators": "Exfoliator",

    # Eye Care
    "Eye Care": "Eye Care",
    "Eye Moisturizer": "Eye Care",
    "Eye Serums": "Eye Care",
    "Eye Mask": "Eye Care",

    # Lip Care
    "Lip Care": "Lip Care",
    "Lip Moisturizer": "Lip Care",
    "Lip Mask": "Lip Care",

    # Sets & Kits
    "Sets": "Sets & Kits",
    "Set": "Sets & Kits",
    "Starter Kits": "Sets & Kits",
}

# 1. Apply mapping
df["Category_Standardized"] = df["Category"].map(category_mapping).fillna(df["Category"])

# 2. Inspect value counts of unified groups
print("--- Standardized Categories ---")
display(df["Category_Standardized"].value_counts())

--- Standardized Categories ---


Category_Standardized
Cleanser           277
Toner              269
Moisturizers       221
Serum & Ampoule    213
Moisturizer        209
Serums             200
Face Masks         179
Sunscreen          132
Eye Care           112
Mask               111
Treatment           88
Essence             83
Lip Care            74
Toners Pads         53
Exfoliator          45
Sets & Kits         24
Name: count, dtype: int64

In [67]:
df.isna().sum()

Brand                      0
Product Name               0
Current Price              0
Original Price             0
Discount (%)               0
URL                        0
Category                   4
Skin Concerns            195
Ingredients               24
Rating                     0
Review_Count               0
Category_Standardized      4
dtype: int64

In [68]:
# 1. Inspect the 24 products still missing ingredients
missing_24 = df[df["Ingredients"].isna()][["Brand", "Product Name", "URL"]]
display(missing_24)

,Brand,Product Name,URL
0,KOREAN SKINCARE,The Tidal Glow Advent Calendar 2026,https://korean-skincare.de/products/korean-ski...
22,COSRX,Master Patch Original Fit,https://korean-skincare.de/products/cosrx-acne...
699,PETITFEE,Super Volume Lip Oil,https://korean-skincare.de/products/petitfee-v...
811,PETITFEE,Oil Blossom Camellia Seed Oil Lip Mask,https://korean-skincare.de/products/petitfee-o...
1185,PYUNKANG YUL,Oil,https://korean-skincare.de/products/oil
1274,PYUNKANG YUL,Clear Spot Patch,https://korean-skincare.de/products/pyunkang-y...
1324,SOME BY MI,Glass Skin Perfect Duo Kit,https://korean-skincare.de/products/some-by-mi...
1407,SKINFOOD,Avocado & Sugar Lip Scrub,https://korean-skincare.de/products/avocado-su...
1435,K-SECRET,SEOUL 1988 Boosting Ball : Collagen 100%,https://korean-skincare.de/products/k-secret-s...
1469,BENTON,Honest Lip Balm,https://korean-skincare.de/products/benton-hon...


In [69]:
# 1. Update Skincare Formulas
skincare_patch = {
    22: "Cellulose Gum, Styrene Isoprene Styrene Block Copolymer, Polyisobutylene, Petroleum Resin, Polyurethane Film, Liquid Paraffin, Tetrakis Methane",
    699: "Hydrogenated Polyisobutene, Simmondsia Chinensis (Jojoba) Seed Oil, Caprylic/Capric Triglyceride, Macadamia Ternifolia Seed Oil, Ethylene/Propylene/Styrene Copolymer, Corylus Avellana (Hazelnut) Seed Oil, Helianthus Annuus (Sunflower) Seed Oil, Prunus Amygdalus Dulcis (Sweet Almond) Oil, Calendula Officinalis Flower Oil, Eclipta Prostrata Extract, Melia Azadirachta Leaf Extract, Moringa Oleifera Seed Oil, Rosa Canina Fruit Oil, Opuntia Ficus-Indica Seed Oil, Anemarrhena Asphodeloides Root Extract, Tocopherol",
    811: "Diisostearyl Malate, Dipentaerythrityl Hexa C5-9 Acid Esters, Hydrogenated Polyisobutene, Microcrystalline Wax, Ceresin, Caprylic/Capric Triglyceride, Beeswax, Curcuma Longa (Turmeric) Root Extract, Petrolatum, Butyrospermum Parkii (Shea Butter), Eclipta Prostrata Extract, Euphorbia Cerifera (Candelilla) Wax, Camellia Japonica Seed Oil, Simmondsia Chinensis (Jojoba) Seed Oil, Limnanthes Alba (Meadowfoam) Seed Oil, Macadamia Integrifolia Seed Oil, Argania Spinosa Kernel Oil, Melia Azadirachta Leaf Extract, Moringa Oleifera Seed Oil, Tocopherol",
    1185: "Simmondsia Chinensis (Jojoba) Seed Oil, Helianthus Annuus (Sunflower) Seed Oil, Butyrospermum Parkii (Shea) Butter, Geranium Maculatum Oil, Rosmarinus Officinalis (Rosemary) Leaf Oil, Citrus Medica Limonum (Lemon) Fruit Oil, Lavandula Angustifolia (Lavender) Oil, Citrus Aurantium Bergamia (Bergamot) Fruit Oil",
    1274: "Cellulose Gum, Hydrogenated C6-20 Polyolefin, Polyisobutene, Styrene/Isoprene Copolymer, Salicylic Acid, Melaleuca Alternifolia (Tea Tree) Leaf Oil, Houttuynia Cordata Extract, Niacinamide, Centella Asiatica Extract",
    1407: "Sucrose, Butyrospermum Parkii (Shea) Butter, Cetyl Ethylhexanoate, Hydrogenated Polyisobutene, Isononyl Isononanoate, Simmondsia Chinensis (Jojoba) Seed Oil, VP/Hexadecene Copolymer, Beeswax, Disteardimonium Hectorite, Euphorbia Cerifera (Candelilla) Wax, Glyceryl Behenate/Eicosadioate, Persea Gratissima (Avocado) Oil, Caramel",
    1435: "Hydrolyzed Collagen, Sodium Hyaluronate, Squalane",
    1469: "Polyglyceryl-2 Triisostearate, Helianthus Annuus (Sunflower) Seed Oil, Polyglyceryl-2 Isostearate/Dimer Dilinoleate Copolymer, Butyrospermum Parkii (Shea) Butter, Sorbitan Olivate, Copernicia Cerifera (Carnauba) Wax, Trihydroxystearin, Monascus Extract, Argania Spinosa Kernel Oil, Camellia Japonica Seed Oil, Camellia Sinensis Seed Oil, Olea Europaea (Olive) Fruit Oil, Macadamia Integrifolia Seed Oil, Simmondsia Chinensis (Jojoba) Seed Oil, Glyceryl Caprylate, Tocopherol",
    1751: "Hydrocolloid Sheet 100% (Cellulose Gum, Polyurethane, Polyisobutylene)",
    1861: "Polyglyceryl-2 Isostearate/Dimer Dilinoleate Copolymer, Polyglyceryl-2 Triisostearate, Caprylic/Capric Triglyceride, Diisostearyl Malate, Ricinus Communis (Castor) Seed Oil, Simmondsia Chinensis (Jojoba) Seed Oil, Helianthus Annuus (Sunflower) Seed Oil, Tocopherol, Fragrance",
    1959: "Collagen (74%), Hydrolyzed Collagen (19%), Glutathione, Hyaluronic Acid",
    1989: "Helianthus Annuus Seed Oil, Simmondsia Chinensis Seed Oil, Orbignya Oleifera Seed Oil, Butyrospermum Parkii Butter, Borago Officinalis Seed Oil, Alpha-Bisabolol, Tocopherol, Citrus Aurantium Amara Flower Oil, Citrus Aurantium Amara Leaf/Twig Oil, Aniba Rosaeodora Wood Oil, Rosa Damascena Flower Oil, Jasminum Officinale Oil",
    2032: "Collagen (74%), Hydrolyzed Collagen (19%), Glutathione, Hyaluronic Acid",
    2065: "Helianthus Annuus (Sunflower) Seed Oil, Caprylic/Capric Triglyceride, Olea Europaea (Olive) Fruit Oil, Macadamia Ternifolia Seed Oil, Simmondsia Chinensis (Jojoba) Seed Oil, Prunus Amygdalus Dulcis (Sweet Almond) Oil, Citrus Aurantium Dulcis (Orange) Peel Oil, Citrus Aurantium Bergamia (Bergamot) Fruit Oil, Citrus Nobilis (Mandarin Orange) Peel Oil",
    2078: "Water, Glycerin, Caprylic/Capric Triglyceride, Hydrogenated Polydecene, Cetearyl Alcohol, Niacinamide, Butylene Glycol, Hydrolyzed Elastin, Collagen Extract, Bakuchiol, Arbutin, Glutathione, Ceramide NP, Hydrolyzed Sponge, Adenosine, Ethylhexylglycerin",
    2086: "Diisostearyl Malate, Dipentaerythrityl Hexa C5-9 Acid Esters, Hydrogenated Polyisobutene, Microcrystalline Wax, Ceresin, Caprylic/Capric Triglyceride, Beeswax, Hippophae Rhamnoides (Sea Buckthorn) Fruit Oil, Moringa Oleifera Seed Oil, Macadamia Ternifolia Seed Oil, Argania Spinosa Kernel Oil, Simmondsia Chinensis (Jojoba) Seed Oil, Limnanthes Alba (Meadowfoam) Seed Oil, Tocopherol",
    2346: "Ricinus Communis (Castor) Seed Oil, Butyrospermum Parkii (Shea) Butter, Beeswax, Persea Gratissima (Avocado) Oil, Macadamia Integrifolia Seed Oil, Argania Spinosa Kernel Oil, Camellia Japonica Seed Oil, Simmondsia Chinensis (Jojoba) Seed Oil, Tocopherol",
    2399: "Collagen (74%), Hydrolyzed Collagen (19%), Glutathione, Hyaluronic Acid",
    2524: "Squalane, Cetyl Ethylhexanoate, C13-15 Alkane, Hydrogenated Polydecene, Hydrogenated Polyisobutene, Candida Bombicola/Glucose/Methyl Rapeseedate Ferment, 4-Terpineol, Curcuma Longa (Turmeric) Root Extract, Helianthus Annuus (Sunflower) Seed Oil, Melia Azadirachta Flower Extract, Melia Azadirachta Leaf Extract, Moringa Oleifera Seed Oil, Ceramide NP"
}

for idx, inci in skincare_patch.items():
    if idx in df.index:
        df.loc[idx, "Ingredients"] = inci

# 2. Mark Multi-Product Sets
set_indices = [0, 1324, 1498]
for idx in set_indices:
    if idx in df.index:
        df.loc[idx, "Ingredients"] = "Multi-Product Set"
        df.loc[idx, "Category"] = "Sets & Kits"

# 3. Handle Makeup
makeup_indices = [2022, 2023]
for idx in makeup_indices:
    if idx in df.index:
        df.loc[idx, "Category"] = "Make-up"

print(f"Remaining null ingredients: {df['Ingredients'].isna().sum()}")

Remaining null ingredients: 2


In [70]:
df["Skin Concerns"].value_counts()

Skin Concerns
All skin types.                                                                                                                                                  260
Dry skin, all skin types.                                                                                                                                         76
Sensitive skin, all skin types.                                                                                                                                   61
Good For Oily Skin,Redness Reducing,Acne Fighting,Brightening                                                                                                     26
Good For Oily Skin,Redness Reducing,Reduces Irritation,Acne Fighting,Brightening                                                                                  25
                                                                                                                                                                .

In [71]:
# Define mapping rules: (Concern, Suitable Skin Types, Keywords)
RULE_MAP = [
    {
        "concern": "Acne & Blemishes",
        "skin_types": ["Oily", "Combination", "Acne-prone"],
        "keywords": ["salicylic acid", "tea tree", "melaleuca", "succinic acid", "zinc pca", "bha"]
    },
    {
        "concern": "Soothing & Redness",
        "skin_types": ["Sensitive", "All Skin Types"],
        "keywords": ["centella", "madecassoside", "asiaticoside", "panthenol", "ceramide", "allantoin", "artemisia", "houttuynia"]
    },
    {
        "concern": "Deep Hydration",
        "skin_types": ["Dry", "Dehydrated", "Normal"],
        "keywords": ["sodium hyaluronate", "hyaluronic acid", "beta-glucan", "polyglutamic acid", "squalane"]
    },
    {
        "concern": "Brightening & Dark Spots",
        "skin_types": ["Dull", "All Skin Types"],
        "keywords": ["niacinamide", "ascorbic acid", "arbutin", "tranexamic acid", "glutathione"]
    },
    {
        "concern": "Anti-Aging & Elasticity",
        "skin_types": ["Mature", "Normal", "Dry"],
        "keywords": ["retinol", "retinal", "bakuchiol", "peptide", "collagen", "adenosine", "ginseng"]
    },
    {
        "concern": "Pores & Sebum",
        "skin_types": ["Oily", "Combination"],
        "keywords": ["kaolin", "bentonite", "charcoal", "volcanic", "witch hazel"]
    }
]

def map_concerns_and_types(inci_str):
    if not isinstance(inci_str, str) or inci_str.strip() == "":
        return [], ["All Skin Types"]
    
    text = inci_str.lower()
    matched_concerns = set()
    matched_skin_types = set()
    
    for rule in RULE_MAP:
        if any(kw in text for kw in rule["keywords"]):
            matched_concerns.add(rule["concern"])
            for st_val in rule["skin_types"]:
                matched_skin_types.add(st_val)
                
    # Defaults if no specific active is flagged
    if not matched_concerns:
        matched_concerns.add("General Maintenance")
    if not matched_skin_types:
        matched_skin_types.add("All Skin Types")
        
    return sorted(list(matched_concerns)), sorted(list(matched_skin_types))

# Apply the tagging function
results = df["Ingredients"].apply(map_concerns_and_types)

df["Skin_Concerns"] = results.apply(lambda x: x[0])
df["Suitable_Skin_Types"] = results.apply(lambda x: x[1])

# For Tableau export, convert lists to clean comma-separated strings
df["Skin_Concerns_Str"] = df["Skin_Concerns"].apply(lambda lst: ", ".join(lst))
df["Suitable_Skin_Types_Str"] = df["Suitable_Skin_Types"].apply(lambda lst: ", ".join(lst))

In [72]:
df

,Brand,Product Name,Current Price,Original Price,Discount (%),URL,Category,Skin Concerns,Ingredients,Rating,Review_Count,Category_Standardized,Skin_Concerns,Suitable_Skin_Types,Skin_Concerns_Str,Suitable_Skin_Types_Str
0,KOREAN SKINCARE,The Tidal Glow Advent Calendar 2026,204.95,204.95,0.0,https://korean-skincare.de/products/korean-ski...,Sets & Kits,NaN,Multi-Product Set,5.00,10.0,NaN,[General Maintenance],[All Skin Types],General Maintenance,All Skin Types
1,BEAUTY OF JOSEON,Relief Sun: Rice + Niacinamide SPF 50+ PA++++,12.61,17.90,29.6,https://korean-skincare.de/products/beauty-of-...,Sunscreen,All skin types aespecially those looking for h...,"Water, Dibutyl Adipate, Propanediol, Diethylam...",4.75,7298.0,Sunscreen,"[Anti-Aging & Elasticity, Brightening & Dark S...","[All Skin Types, Dry, Dull, Mature, Normal]","Anti-Aging & Elasticity, Brightening & Dark Spots","All Skin Types, Dry, Dull, Mature, Normal"
2,DR.ALTHEA,345 Relief Cream,17.82,22.95,22.4,https://korean-skincare.de/products/dr-althea-...,Moisturizers,"Acne-prone skin, sensitive skin, redness-prone...","Aqua, Melaleuca Alternifolia Leaf Water, Propa...",4.60,2644.0,Moisturizers,"[Acne & Blemishes, Brightening & Dark Spots, D...","[Acne-prone, All Skin Types, Combination, Dehy...","Acne & Blemishes, Brightening & Dark Spots, De...","Acne-prone, All Skin Types, Combination, Dehyd..."
3,BEAUTY OF JOSEON,Dynasty Cream,16.85,23.95,29.6,https://korean-skincare.de/products/beauty-of-...,General Moisturizer,"Good For Oily Skin,Redness Reducing,Reduces Ir...","Water,Glycerin,Butylene Glycol,Niacinamide,Dip...",4.62,2645.0,Moisturizer,"[Anti-Aging & Elasticity, Brightening & Dark S...","[All Skin Types, Dehydrated, Dry, Dull, Mature...","Anti-Aging & Elasticity, Brightening & Dark Sp...","All Skin Types, Dehydrated, Dry, Dull, Mature,..."
4,ANUA,Heartleaf Quercetinol Pore Deep Cleansing Foam,12.38,16.45,24.7,https://korean-skincare.de/products/anua-heart...,Cleansers,"Acne prone skin, oily skin, combination skin","Houttuynia Cordata Flower/Leaf/Stem Water, Gly...",4.55,1356.0,Cleanser,"[Acne & Blemishes, Deep Hydration, Soothing & ...","[Acne-prone, All Skin Types, Combination, Dehy...","Acne & Blemishes, Deep Hydration, Soothing & R...","Acne-prone, All Skin Types, Combination, Dehyd..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2708,DR. CEURACLE,Hyal Reyouth Lifting Mask Set,1.95,1.95,0.0,https://korean-skincare.de/products/dr-ceuracl...,Sheet Mask,"Good For Oily Skin,Redness Reducing,Reduces Ir...","Water,Glycerin,Dipropylene Glycol,Alcohol,Peg/...",4.46,13.0,Mask,"[Anti-Aging & Elasticity, Deep Hydration, Soot...","[All Skin Types, Dehydrated, Dry, Mature, Norm...","Anti-Aging & Elasticity, Deep Hydration, Sooth...","All Skin Types, Dehydrated, Dry, Mature, Norma..."
2709,AXIS-Y,Green Vital Energy Complex Sheet Mask,2.45,2.45,0.0,https://korean-skincare.de/products/axis-y-gre...,Face Masks,NaN,"Artemisia Capillaris Extract, Water, Dipropyle...",4.86,7.0,Face Masks,"[Deep Hydration, Soothing & Redness]","[All Skin Types, Dehydrated, Dry, Normal, Sens...","Deep Hydration, Soothing & Redness","All Skin Types, Dehydrated, Dry, Normal, Sensi..."
2710,HYGGEE,Active Red Flower Mask,1.71,2.95,42.0,https://korean-skincare.de/products/hyggee-act...,Face Masks,All skin types.,"Water, Glycerin, Dipropylene Glycol, Cetyl Eth...",4.58,12.0,Face Masks,"[Anti-Aging & Elasticity, Brightening & Dark S...","[All Skin Types, Dehydrated, Dry, Dull, Mature...","Anti-Aging & Elasticity, Brightening & Dark Sp...","All Skin Types, Dehydrated, Dry, Dull, Mature,..."
2711,GOODAL,Green Tangerine Vitamin C Serum Mask,2.95,2.95,0.0,https://korean-skincare.de/products/goodal-gre...,Serum,"Hydrating,Good For Oily Skin,Redness Reducing,...","Citrus Tangerina Extract,Butylene Glycol,Niaci...",5.00,8.0,Serum & Ampoule,"[Anti-Aging & Elasticity, Brightening & Dark S...","[All Skin Types, Dehydrated, Dry, Dull, Mature...","Anti-Aging & Elasticity, Brightening & Dark Sp...","All 

In [73]:
df["Category_Standardized"].value_counts()

Category_Standardized
Cleanser           277
Toner              269
Moisturizers       221
Serum & Ampoule    213
Moisturizer        209
Serums             200
Face Masks         179
Sunscreen          132
Eye Care           112
Mask               111
Treatment           88
Essence             83
Lip Care            74
Toners Pads         53
Exfoliator          45
Sets & Kits         24
Name: count, dtype: int64

In [74]:
# 1. Define all variations of sets and kits in the Category column
set_categories = ["Sets", "Starter Kits", "Sets & Kits", "Set"]

# 2. Drop those rows and reset the index
df = df[~df["Category"].isin(set_categories)].reset_index(drop=True)

# 3. Verify the drop
print(f"Remaining products: {len(df)}")
print(df["Category"].value_counts())

Remaining products: 2269
Category
Moisturizers           221
Serums                 200
Face Masks             179
General Moisturizer    169
Toner                  169
Serum                  166
Sunscreen              132
Cleansers              122
Face Cleanser          117
Toners                 100
Eye Care                62
Essence                 53
Toners Pads             53
Facial Treatment        51
Sheet Mask              49
Lip Care                49
Ampoules                47
Wet Mask                39
Makeup Remover          35
Eye Moisturizer         33
Exfoliator              32
Essences                30
Spot Patches            20
Overnight Mask          20
Lip Moisturizer         15
Eye Mask                13
Exfoliators             13
Night Moisturizer       12
Treatments              11
Lip Mask                 9
Day Moisturizer          9
Emulsion                 8
Moisturisers             8
Eye Serums               4
Multi Balms              4
Masks                

In [75]:
df.isna().sum()

Brand                        0
Product Name                 0
Current Price                0
Original Price               0
Discount (%)                 0
URL                          0
Category                     2
Skin Concerns              188
Ingredients                  2
Rating                       0
Review_Count                 0
Category_Standardized        3
Skin_Concerns                0
Suitable_Skin_Types          0
Skin_Concerns_Str            0
Suitable_Skin_Types_Str      0
dtype: int64

In [76]:
df_clean = df.dropna().reset_index(drop=True)

In [77]:
df_clean.isna().sum()

Brand                      0
Product Name               0
Current Price              0
Original Price             0
Discount (%)               0
URL                        0
Category                   0
Skin Concerns              0
Ingredients                0
Rating                     0
Review_Count               0
Category_Standardized      0
Skin_Concerns              0
Suitable_Skin_Types        0
Skin_Concerns_Str          0
Suitable_Skin_Types_Str    0
dtype: int64

In [78]:
df_clean

,Brand,Product Name,Current Price,Original Price,Discount (%),URL,Category,Skin Concerns,Ingredients,Rating,Review_Count,Category_Standardized,Skin_Concerns,Suitable_Skin_Types,Skin_Concerns_Str,Suitable_Skin_Types_Str
0,BEAUTY OF JOSEON,Relief Sun: Rice + Niacinamide SPF 50+ PA++++,12.61,17.90,29.6,https://korean-skincare.de/products/beauty-of-...,Sunscreen,All skin types aespecially those looking for h...,"Water, Dibutyl Adipate, Propanediol, Diethylam...",4.75,7298.0,Sunscreen,"[Anti-Aging & Elasticity, Brightening & Dark S...","[All Skin Types, Dry, Dull, Mature, Normal]","Anti-Aging & Elasticity, Brightening & Dark Spots","All Skin Types, Dry, Dull, Mature, Normal"
1,DR.ALTHEA,345 Relief Cream,17.82,22.95,22.4,https://korean-skincare.de/products/dr-althea-...,Moisturizers,"Acne-prone skin, sensitive skin, redness-prone...","Aqua, Melaleuca Alternifolia Leaf Water, Propa...",4.60,2644.0,Moisturizers,"[Acne & Blemishes, Brightening & Dark Spots, D...","[Acne-prone, All Skin Types, Combination, Dehy...","Acne & Blemishes, Brightening & Dark Spots, De...","Acne-prone, All Skin Types, Combination, Dehyd..."
2,BEAUTY OF JOSEON,Dynasty Cream,16.85,23.95,29.6,https://korean-skincare.de/products/beauty-of-...,General Moisturizer,"Good For Oily Skin,Redness Reducing,Reduces Ir...","Water,Glycerin,Butylene Glycol,Niacinamide,Dip...",4.62,2645.0,Moisturizer,"[Anti-Aging & Elasticity, Brightening & Dark S...","[All Skin Types, Dehydrated, Dry, Dull, Mature...","Anti-Aging & Elasticity, Brightening & Dark Sp...","All Skin Types, Dehydrated, Dry, Dull, Mature,..."
3,ANUA,Heartleaf Quercetinol Pore Deep Cleansing Foam,12.38,16.45,24.7,https://korean-skincare.de/products/anua-heart...,Cleansers,"Acne prone skin, oily skin, combination skin","Houttuynia Cordata Flower/Leaf/Stem Water, Gly...",4.55,1356.0,Cleanser,"[Acne & Blemishes, Deep Hydration, Soothing & ...","[Acne-prone, All Skin Types, Combination, Dehy...","Acne & Blemishes, Deep Hydration, Soothing & R...","Acne-prone, All Skin Types, Combination, Dehyd..."
4,BEAUTY OF JOSEON,Revive Eye Serum : Ginseng + Retinal,13.00,17.95,27.6,https://korean-skincare.de/products/beauty-of-...,Eye Moisturizer,"Good For Oily Skin,Redness Reducing,Skin Textu...","Water,Panax Ginseng Root Extract,Glycerin,Dipr...",4.53,2114.0,Eye Care,"[Anti-Aging & Elasticity, Brightening & Dark S...","[All Skin Types, Dehydrated, Dry, Dull, Mature...","Anti-Aging & Elasticity, Brightening & Dark Sp...","All Skin Types, Dehydrated, Dry, Dull, Mature,..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2073,ILSO,Daily Moisture Pudding Cream,27.95,27.95,0.0,https://korean-skincare.de/products/ilso-daily...,Moisturizers,"Dry skin, all skin types.","Water, Butylene Glycol, Propanediol, Isotridec...",0.00,0.0,Moisturizers,"[Deep Hydration, Soothing & Redness]","[All Skin Types, Dehydrated, Dry, Normal, Sens...","Deep Hydration, Soothing & Redness","All Skin Types, Dehydrated, Dry, Normal, Sensi..."
2074,ILSO,Natural Mild Cleansing Oil,25.95,25.95,0.0,https://korean-skincare.de/products/ilso-natur...,Cleansers,All skin types.,"Ethylhexyl Palmitate, Cetyl Ethylhexanoate, So...",5.00,1.0,Cleanser,[Acne & Blemishes],"[Acne-prone, Combination, Oily]",Acne & Blemishes,"Acne-prone, Combination, Oily"
2075,DR. CEURACLE,Hyal Reyouth Lifting Mask Set,1.95,1.95,0.0,https://korean-skincare.de/products/dr-ceuracl...,Sheet Mask,"Good For Oily Skin,Redness Reducing,Reduces Ir...","Water,Glycerin,Dipropylene Glycol,Alcohol,Peg/...",4.46,13.0,Mask,"[Anti-Aging & Elasticity, Deep Hydration, Soot...","[All Skin Types, Dehydrated, Dry, Mature, Norm...","Anti-Aging & Elasticity, Deep Hydration, Sooth...","All Skin Types, Dehydrated, Dry, Mature, Norma..."
2076,HYGGEE,Active Red Flower Mask,1.71,2.95,42.0,https://korean-skincare.de/products/hyggee-act...,Face Masks,All skin types.,"Water, Glycerin, Dipropylene Glycol, Cetyl Eth...",4.58,12.0,Face Masks,"[Anti-Aging & Elasticity, Brightening & Dark S...","[All Skin Types,

In [92]:
df_clean["Category_Standardized"].value_counts()

Category_Standardized
Moisturizer    401
Serum          396
Toner          306
Cleanser       264
Mask           242
Sunscreen      128
Eye Care        95
Treatment       87
Essence         74
Exfoliator      43
Lip Care        42
Name: count, dtype: int64

In [91]:
# Standardize categories cleanly in one pass
df_clean["Category_Standardized"] = df_clean["Category_Standardized"].replace({
    "Serum & Ampoule": "Serum",
    "Serums": "Serum",
    "Ampoules": "Serum",
    "Moisturizers": "Moisturizer",
    "General Moisturizer": "Moisturizer",
    "Face Masks": "Mask",
    "Masks": "Mask",
    "Sheet Mask": "Mask",
    "Toners": "Toner",
    "Toners Pads": "Toner",
    "Cleansers": "Cleanser",
    "Face Cleanser": "Cleanser"
})
# Verify the count (should be ~408 products)
print(df_clean["Category_Standardized"].value_counts())

Category_Standardized
Moisturizer    401
Serum          396
Toner          306
Cleanser       264
Mask           242
Sunscreen      128
Eye Care        95
Treatment       87
Essence         74
Exfoliator      43
Lip Care        42
Name: count, dtype: int64


In [93]:
df_clean

,Brand,Product Name,Current Price,Original Price,Discount (%),URL,Category,Skin Concerns,Ingredients,Rating,...,Skin_Type_Dry,Skin_Type_Combination,Skin_Type_Sensitive,Skin_Type_Normal,Skin_Type_All,Concern_Acne,Concern_Aging,Concern_Brightening,Concern_Redness,Concern_Hydration
0,BEAUTY OF JOSEON,Relief Sun: Rice + Niacinamide SPF 50+ PA++++,12.61,17.90,29.6,https://korean-skincare.de/products/beauty-of-...,Sunscreen,All skin types aespecially those looking for h...,"Water, Dibutyl Adipate, Propanediol, Diethylam...",4.75,...,0,0,0,0,1,0,0,1,0,0
1,DR.ALTHEA,345 Relief Cream,17.82,22.95,22.4,https://korean-skincare.de/products/dr-althea-...,Moisturizers,"Acne-prone skin, sensitive skin, redness-prone...","Aqua, Melaleuca Alternifolia Leaf Water, Propa...",4.60,...,0,0,1,0,0,1,0,0,1,1
2,BEAUTY OF JOSEON,Dynasty Cream,16.85,23.95,29.6,https://korean-skincare.de/products/beauty-of-...,General Moisturizer,"Good For Oily Skin,Redness Reducing,Reduces Ir...","Water,Glycerin,Butylene Glycol,Niacinamide,Dip...",4.62,...,0,0,0,0,0,1,1,1,1,0
3,ANUA,Heartleaf Quercetinol Pore Deep Cleansing Foam,12.38,16.45,24.7,https://korean-skincare.de/products/anua-heart...,Cleansers,"Acne prone skin, oily skin, combination skin","Houttuynia Cordata Flower/Leaf/Stem Water, Gly...",4.55,...,0,1,0,0,0,1,0,0,0,0
4,BEAUTY OF JOSEON,Revive Eye Serum : Ginseng + Retinal,13.00,17.95,27.6,https://korean-skincare.de/products/beauty-of-...,Eye Moisturizer,"Good For Oily Skin,Redness Reducing,Skin Textu...","Water,Panax Ginseng Root Extract,Glycerin,Dipr...",4.53,...,1,0,0,0,0,1,1,1,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2073,ILSO,Daily Moisture Pudding Cream,27.95,27.95,0.0,https://korean-skincare.de/products/ilso-daily...,Moisturizers,"Dry skin, all skin types.","Water, Butylene Glycol, Propanediol, Isotridec...",0.00,...,1,0,0,0,1,0,0,0,0,1
2074,ILSO,Natural Mild Cleansing Oil,25.95,25.95,0.0,https://korean-skincare.de/products/ilso-natur...,Cleansers,All skin types.,"Ethylhexyl Palmitate, Cetyl Ethylhexanoate, So...",5.00,...,0,0,0,0,1,0,0,0,0,0
2075,DR. CEURACLE,Hyal Reyouth Lifting Mask Set,1.95,1.95,0.0,https://korean-skincare.de/products/dr-ceuracl...,Sheet Mask,"Good For Oily Skin,Redness Reducing,Reduces Ir...","Water,Glycerin,Dipropylene Glycol,Alcohol,Peg/...",4.46,...,0,0,0,0,0,0,1,1,1,0
2076,HYGGEE,Active Red Flower Mask,1.71,2.95,42.0,https://korean-skincare.de/products/hyggee-act...,Face Masks,All skin types.,"Water, Glycerin, Dipropylene Glycol, Cetyl Eth...",4.58,...,0,0,0,0,1,0,0,0,0,0


In [94]:
# 2. Fix chemical numbers (e.g. 1,2-Hexanediol) so commas inside ingredient names don't break splitting
corrected_ingredients = df_clean["Ingredients"].dropna().str.replace(r"\b1,\s*2-", "1.2-", regex=True)

# 3. Split, clean, and count individual ingredients
all_ingredients = (
    corrected_ingredients
    .str.split(",")
    .explode()
    .str.strip()
    .str.title()
)

print("=== Top 20 Most Common Ingredients ===")
print(all_ingredients.value_counts().head(20))

# 4. Tag Hero Actives accurately
actives = {
    "Hyaluronic Acid": "hyaluronic acid|sodium hyaluronate|hydrolyzed hyaluronic acid",
    "Panthenol": "panthenol",
    "Niacinamide": "niacinamide",
    "Adenosine": "adenosine",
    "Centella Asiatica": "centella",
    "Ceramides": "ceramide",
    "Tea Tree Oil": "melaleuca alternifolia|tea tree",
    "Salicylic Acid (BHA)": "salicylic acid",
    "Retinol / Retinal": "retinol|retinal",
    "Snail Mucin": "snail secretion filtrate"
}

active_counts = {name: df_clean["Ingredients"].str.contains(kw, case=False, na=False).sum() for name, kw in actives.items()}
active_df = pd.DataFrame(list(active_counts.items()), columns=["Active", "Count"]).sort_values(by="Count", ascending=False)

print("\n=== Hero Active Product Counts ===")
print(active_df.to_string(index=False))

=== Top 20 Most Common Ingredients ===
Ingredients
Water                           1574
Butylene Glycol                 1574
1.2-Hexanediol                  1556
Glycerin                        1515
Ethylhexylglycerin              1294
Sodium Hyaluronate               956
Disodium Edta                    926
Niacinamide                      774
Panthenol                        761
Adenosine                        735
Propanediol                      661
Allantoin                        647
Xanthan Gum                      615
Carbomer                         598
Caprylyl Glycol                  574
Hydrogenated Lecithin            561
Tocopherol                       557
Caprylic/Capric Triglyceride     540
Dipropylene Glycol               539
Tromethamine                     531
Name: count, dtype: int64

=== Hero Active Product Counts ===
              Active  Count
     Hyaluronic Acid   1098
         Niacinamide    804
           Panthenol    779
           Adenosine    740
   Cent

In [95]:
ingredient_groups = {
    "Has_Niacinamide": "niacinamide",
    "Has_Hyaluronic_Acid": "hyaluronic acid|sodium hyaluronate|hydrolyzed hyaluronic acid",
    "Has_Centella": "centella|madecassoside|asiaticoside",
    "Has_Ceramides": "ceramide",
    "Has_Retinol": "retinol|retinal|bakuchiol",
    "Has_BHA_AHA": "salicylic acid|glycolic acid|lactic acid|betaine salicylate|gluconolactone",
    "Has_Panthenol": "panthenol",
    "Has_Snail_Mucin": "snail secretion filtrate",
    "Has_Vitamin_C": "ascorbic acid|ascorbyl glucoside",
    "Has_Peptides": "peptide|palmitoyl",
}

for col_name, pattern in ingredient_groups.items():
    df_clean[col_name] = (
        df_clean["Ingredients"].str.contains(pattern, case=False, na=False).astype(int)
    )

In [96]:
df_clean

,Brand,Product Name,Current Price,Original Price,Discount (%),URL,Category,Skin Concerns,Ingredients,Rating,...,Skin_Type_Dry,Skin_Type_Combination,Skin_Type_Sensitive,Skin_Type_Normal,Skin_Type_All,Concern_Acne,Concern_Aging,Concern_Brightening,Concern_Redness,Concern_Hydration
0,BEAUTY OF JOSEON,Relief Sun: Rice + Niacinamide SPF 50+ PA++++,12.61,17.90,29.6,https://korean-skincare.de/products/beauty-of-...,Sunscreen,All skin types aespecially those looking for h...,"Water, Dibutyl Adipate, Propanediol, Diethylam...",4.75,...,0,0,0,0,1,0,0,1,0,0
1,DR.ALTHEA,345 Relief Cream,17.82,22.95,22.4,https://korean-skincare.de/products/dr-althea-...,Moisturizers,"Acne-prone skin, sensitive skin, redness-prone...","Aqua, Melaleuca Alternifolia Leaf Water, Propa...",4.60,...,0,0,1,0,0,1,0,0,1,1
2,BEAUTY OF JOSEON,Dynasty Cream,16.85,23.95,29.6,https://korean-skincare.de/products/beauty-of-...,General Moisturizer,"Good For Oily Skin,Redness Reducing,Reduces Ir...","Water,Glycerin,Butylene Glycol,Niacinamide,Dip...",4.62,...,0,0,0,0,0,1,1,1,1,0
3,ANUA,Heartleaf Quercetinol Pore Deep Cleansing Foam,12.38,16.45,24.7,https://korean-skincare.de/products/anua-heart...,Cleansers,"Acne prone skin, oily skin, combination skin","Houttuynia Cordata Flower/Leaf/Stem Water, Gly...",4.55,...,0,1,0,0,0,1,0,0,0,0
4,BEAUTY OF JOSEON,Revive Eye Serum : Ginseng + Retinal,13.00,17.95,27.6,https://korean-skincare.de/products/beauty-of-...,Eye Moisturizer,"Good For Oily Skin,Redness Reducing,Skin Textu...","Water,Panax Ginseng Root Extract,Glycerin,Dipr...",4.53,...,1,0,0,0,0,1,1,1,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2073,ILSO,Daily Moisture Pudding Cream,27.95,27.95,0.0,https://korean-skincare.de/products/ilso-daily...,Moisturizers,"Dry skin, all skin types.","Water, Butylene Glycol, Propanediol, Isotridec...",0.00,...,1,0,0,0,1,0,0,0,0,1
2074,ILSO,Natural Mild Cleansing Oil,25.95,25.95,0.0,https://korean-skincare.de/products/ilso-natur...,Cleansers,All skin types.,"Ethylhexyl Palmitate, Cetyl Ethylhexanoate, So...",5.00,...,0,0,0,0,1,0,0,0,0,0
2075,DR. CEURACLE,Hyal Reyouth Lifting Mask Set,1.95,1.95,0.0,https://korean-skincare.de/products/dr-ceuracl...,Sheet Mask,"Good For Oily Skin,Redness Reducing,Reduces Ir...","Water,Glycerin,Dipropylene Glycol,Alcohol,Peg/...",4.46,...,0,0,0,0,0,0,1,1,1,0
2076,HYGGEE,Active Red Flower Mask,1.71,2.95,42.0,https://korean-skincare.de/products/hyggee-act...,Face Masks,All skin types.,"Water, Glycerin, Dipropylene Glycol, Cetyl Eth...",4.58,...,0,0,0,0,1,0,0,0,0,0


In [97]:
skin_type_groups = {
    "Skin_Type_Oily": "oily",
    "Skin_Type_Dry": "dry",
    "Skin_Type_Combination": "combination",
    "Skin_Type_Sensitive": "sensitive",
    "Skin_Type_Normal": "normal",
    "Skin_Type_All": "all skin types|all skin",
    "Concern_Acne": "acne|blemish|breakout|pimple",
    "Concern_Aging": "aging|wrinkle|elasticity|firmness|fine lines",
    "Concern_Brightening": "brighten|glow|dark spot|pigmentation|tone|melasma",
    "Concern_Redness": "redness|sooth|calm|irritat|rosacea",
    "Concern_Hydration": "hydrat|dry|moistur|dehydrat",
}

for col_name, pattern in skin_type_groups.items():
    df_clean[col_name] = (
        df_clean["Skin Concerns"].str.contains(pattern, case=False, na=False).astype(int)
    )

In [99]:
df_clean

,Brand,Product Name,Current Price,Original Price,Discount (%),URL,Category,Skin Concerns,Ingredients,Rating,...,Skin_Type_Dry,Skin_Type_Combination,Skin_Type_Sensitive,Skin_Type_Normal,Skin_Type_All,Concern_Acne,Concern_Aging,Concern_Brightening,Concern_Redness,Concern_Hydration
0,BEAUTY OF JOSEON,Relief Sun: Rice + Niacinamide SPF 50+ PA++++,12.61,17.90,29.6,https://korean-skincare.de/products/beauty-of-...,Sunscreen,All skin types aespecially those looking for h...,"Water, Dibutyl Adipate, Propanediol, Diethylam...",4.75,...,0,0,0,0,1,0,0,1,0,0
1,DR.ALTHEA,345 Relief Cream,17.82,22.95,22.4,https://korean-skincare.de/products/dr-althea-...,Moisturizers,"Acne-prone skin, sensitive skin, redness-prone...","Aqua, Melaleuca Alternifolia Leaf Water, Propa...",4.60,...,0,0,1,0,0,1,0,0,1,1
2,BEAUTY OF JOSEON,Dynasty Cream,16.85,23.95,29.6,https://korean-skincare.de/products/beauty-of-...,General Moisturizer,"Good For Oily Skin,Redness Reducing,Reduces Ir...","Water,Glycerin,Butylene Glycol,Niacinamide,Dip...",4.62,...,0,0,0,0,0,1,1,1,1,0
3,ANUA,Heartleaf Quercetinol Pore Deep Cleansing Foam,12.38,16.45,24.7,https://korean-skincare.de/products/anua-heart...,Cleansers,"Acne prone skin, oily skin, combination skin","Houttuynia Cordata Flower/Leaf/Stem Water, Gly...",4.55,...,0,1,0,0,0,1,0,0,0,0
4,BEAUTY OF JOSEON,Revive Eye Serum : Ginseng + Retinal,13.00,17.95,27.6,https://korean-skincare.de/products/beauty-of-...,Eye Moisturizer,"Good For Oily Skin,Redness Reducing,Skin Textu...","Water,Panax Ginseng Root Extract,Glycerin,Dipr...",4.53,...,1,0,0,0,0,1,1,1,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2073,ILSO,Daily Moisture Pudding Cream,27.95,27.95,0.0,https://korean-skincare.de/products/ilso-daily...,Moisturizers,"Dry skin, all skin types.","Water, Butylene Glycol, Propanediol, Isotridec...",0.00,...,1,0,0,0,1,0,0,0,0,1
2074,ILSO,Natural Mild Cleansing Oil,25.95,25.95,0.0,https://korean-skincare.de/products/ilso-natur...,Cleansers,All skin types.,"Ethylhexyl Palmitate, Cetyl Ethylhexanoate, So...",5.00,...,0,0,0,0,1,0,0,0,0,0
2075,DR. CEURACLE,Hyal Reyouth Lifting Mask Set,1.95,1.95,0.0,https://korean-skincare.de/products/dr-ceuracl...,Sheet Mask,"Good For Oily Skin,Redness Reducing,Reduces Ir...","Water,Glycerin,Dipropylene Glycol,Alcohol,Peg/...",4.46,...,0,0,0,0,0,0,1,1,1,0
2076,HYGGEE,Active Red Flower Mask,1.71,2.95,42.0,https://korean-skincare.de/products/hyggee-act...,Face Masks,All skin types.,"Water, Glycerin, Dipropylene Glycol, Cetyl Eth...",4.58,...,0,0,0,0,1,0,0,0,0,0


In [100]:
df_clean.to_csv("korean_skincare_master.csv", index=False, encoding="utf-8-sig")

In [101]:
# exporting for Tableau

In [102]:
# Load your master dataset
df = pd.read_csv("korean_skincare_master.csv")

# Columns to drop for Tableau cleanliness
columns_to_drop = [
    "Skin Concerns",  # Replaced by binary concern flags and Concern_Str
    "URL",  # Unnecessary for visual charts unless using web actions
    "Category",  # Replaced by Category_Standardized
]

# Drop columns if they exist in the dataframe
df_tableau = df.drop(
    columns=[col for col in columns_to_drop if col in df.columns]
)

# Save as Tableau-ready file
df_tableau.to_csv(
    "korean_skincare_tableau_ready.csv", index=False, encoding="utf-8-sig"
)
print(f"Tableau-ready export complete! Shape: {df_tableau.shape}")

Tableau-ready export complete! Shape: (2078, 34)
